In [ ]:
# Install dependencies if needed (uncomment in Databricks)
# %pip install -r ../requirements.txt
# dbutils.library.restartPython()

In [ ]:
import sys
import time
from pprint import pprint

sys.path.insert(0, '..')

from multiAgentSystem.agents.reasoning import reasoning_node
from experiments.mlflow_setup import (
    setup_experiment,
    create_agent_run,
    log_agent_metrics,
    log_state_snapshot,
    enable_autologging
)
from experiments.mock_data import REASONING_TEST_STATES, get_test_state

print("✓ Imports successful")

In [ ]:
# Setup MLflow experiment
enable_autologging()
experiment_id = setup_experiment("reasoning")
print(f"Experiment ID: {experiment_id}")

In [ ]:
def run_reasoning_test(scenario_name: str, verbose: bool = True):
    """
    Run a single reasoning test scenario with MLflow tracking.
    """
    scenario = get_test_state("reasoning", scenario_name)
    test_state = scenario["state"].copy()
    expected = scenario["expected"]
    
    with create_agent_run("reasoning", scenario=scenario_name) as run:
        log_state_snapshot(test_state, prefix="input")
        
        start_time = time.time()
        try:
            result = reasoning_node(test_state)
            success = True
            error = None
        except Exception as e:
            result = {"error": str(e)}
            success = False
            error = str(e)
        latency_ms = (time.time() - start_time) * 1000
        
        # Log metrics
        log_agent_metrics(
            latency_ms=latency_ms,
            success=success,
            additional_metrics={
                "input_iteration": test_state.get("iteration", 0),
                "input_analyze_loops": test_state.get("analyze_parse_loops", 0),
                "output_confidence": result.get("confidence", 0.0) if success else 0.0,
                "hypotheses_count": len(result.get("hypotheses", [])) if success else 0,
            }
        )
        
        log_state_snapshot(result, prefix="output")
        
        # Check expected outcomes
        passed = True
        if "next_action" in expected:
            if result.get("next_action") != expected["next_action"]:
                passed = False
        if "last_status" in expected:
            if result.get("last_status") != expected["last_status"]:
                passed = False
        if expected.get("has_hypotheses"):
            if not result.get("hypotheses"):
                passed = False
        if expected.get("has_draft"):
            if not result.get("draft") or not result["draft"].get("problem"):
                passed = False
        
        import mlflow
        mlflow.log_metric("test_passed", 1.0 if passed else 0.0)
        
        if verbose:
            status = "✅ PASSED" if passed else "❌ FAILED"
            print(f"\n{status} - {scenario_name}")
            print(f"  Description: {scenario['description']}")
            print(f"  Latency: {latency_ms:.2f}ms")
            print(f"  last_status: {result.get('last_status')}")
            print(f"  next_action: {result.get('next_action')}")
            print(f"  Hypotheses: {len(result.get('hypotheses', []))}")
            if result.get("draft"):
                print(f"  Draft problem: {result['draft'].get('problem', '')[:80]}...")
                print(f"  Confidence: {result.get('confidence', 0):.2f}")
        
        return result, passed, latency_ms

## Test 1: No Evidence

Initial state with no evidence. Should generate hypotheses and request analyzer.

In [ ]:
result_1, passed_1, latency_1 = run_reasoning_test("no_evidence")

## Test 2: Partial Evidence

Some evidence collected but not sufficient. Should request more.

In [ ]:
result_2, passed_2, latency_2 = run_reasoning_test("partial_evidence")

## Test 3: Sufficient Evidence

Rich evidence available. Should generate RCA draft.

In [ ]:
result_3, passed_3, latency_3 = run_reasoning_test("sufficient_evidence")

## Test 4: Max Loops Reached

Maximum analyze-parse loops reached. Should force summarization.

In [ ]:
result_4, passed_4, latency_4 = run_reasoning_test("max_loops_reached")

## Summary

In [ ]:
print("=" * 60)
print("REASONING AGENT TEST SUMMARY")
print("=" * 60)

tests = [
    ("no_evidence", passed_1, latency_1),
    ("partial_evidence", passed_2, latency_2),
    ("sufficient_evidence", passed_3, latency_3),
    ("max_loops_reached", passed_4, latency_4),
]

total_passed = sum(1 for _, passed, _ in tests if passed)
avg_latency = sum(lat for _, _, lat in tests) / len(tests)

for name, passed, latency in tests:
    status = "✅" if passed else "❌"
    print(f"  {status} {name}: {latency:.2f}ms")

print("=" * 60)
print(f"Total: {total_passed}/{len(tests)} passed")
print(f"Average latency: {avg_latency:.2f}ms")
print("=" * 60)